# Düzenlileştirme

**Titanic** yolcularının hayatta kalma şansını etkileyen faktörlere dair anlayışımızı geliştirelim
- Yorumlaması kolay olan lojistik sınıflandırıcıları kullanacağız
- Bunu daha önce "Karar Bilimi - Lojistik Regresyon" dersinde statsmodels ile yapmıştık
- Hangi özelliklerin alakasız olduğunu / genelleştirilemediğini tespit etmek için `p-değerleri` ve istatistiksel varsayımlar kullanıyorduk
- Bu sefer, eksik/aşırı öğrenme kriterlerine dayalı olarak alakalı/alakasız özellikleri tespit etmek için `düzenlileştirme` kullanacağız
- **Amacımız `L1` ve `L2` cezalarını karşılaştırmak**

## 1. Veriyi sizin için yüklüyor ve ön işleme tabi tutuyoruz

In [1]:
import pandas as pd
import numpy as np

In [4]:
data = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/ML_titanic_dataset_encoded.csv")

# the dataset is already one-hot-encoded
data.head()

,survived,pclass,age,sibsp,parch,fare,sex_female,class_First,class_Third,who_child,embark_town_Cherbourg,embark_town_Queenstown,embark_town_Southampton
0,0,3,22.0,1,0,7.2500,0,0,1,0,0,0,1
1,1,1,38.0,1,0,71.2833,1,1,0,0,1,0,0
2,1,3,26.0,0,0,7.9250,1,0,1,0,0,0,1
3,1,1,35.0,1,0,53.1000,1,1,0,0,0,0,1
4,0,3,35.0,0,0,8.0500,0,0,1,0,0,0,1


In [5]:
# We build X and y

y = data["survived"]
X = data.drop(columns=["survived"])
X.head()

,pclass,age,sibsp,parch,fare,sex_female,class_First,class_Third,who_child,embark_town_Cherbourg,embark_town_Queenstown,embark_town_Southampton
0,3,22.0,1,0,7.2500,0,0,1,0,0,0,1
1,1,38.0,1,0,71.2833,1,1,0,0,1,0,0
2,3,26.0,0,0,7.9250,1,0,1,0,0,0,1
3,1,35.0,1,0,53.1000,1,1,0,0,0,0,1
4,3,35.0,0,0,8.0500,0,0,1,0,0,0,1


In [6]:
# We MinMaxScale our features for you
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler().fit(X)
X_scaled = scaler.transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
X.shape

(714, 12)

## 2. Düzenlileştirme olmadan Lojistik Regresyon

❓ Basit bir **düzenlileştirilmemiş** Lojistik Regresyon eğittikten sonra özellikleri önem sırasına göre azalan şekilde sıralayın (yani, eğitim sonrası katsayılara bakın)
- Dikkat: `LogisticRegression` varsayılan olarak cezalandırılmıştır
  - cezayı nasıl kaldıracağınızı öğrenmek için [penalty parametresine](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) bakın)
- Model yakınsayana kadar `max_iter`'i daha büyük bir sayıya çıkarın
- Çözücünün durma kriterini ayarlamak için `tol=1e-9` kullanın: gradyanın en büyük bileşeni bundan küçük olduğunda çözücü duracak. Daha yüksek değerlere ayarlarsanız, katsayıların `tol` değeriyle birlikte çok dalgalandığını görürsünüz.

<details>
    <summary>İpucu</summary>
    <img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/05-ML/05-Model-Tuning/model_selection.png" alt="penalizing a regression" width="500">
</details>

In [7]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(penalty=None, max_iter=10000, tol=1e-9)
log_model.fit(X_scaled, y)

coefs_unreg = pd.Series(log_model.coef_[0], index=X.columns)
coefs_unreg.reindex(coefs_unreg.abs().sort_values(ascending=False).index)

embark_town_Queenstown    -21.346844
embark_town_Southampton   -20.951437
embark_town_Cherbourg     -20.650067
pclass                      5.240985
class_Third                -3.803688
class_First                 3.707294
sex_female                  2.671878
sibsp                      -2.476885
age                        -2.196127
fare                        1.360204
who_child                   1.336358
parch                      -0.894275
dtype: float64

❓`sex_female` katsayısının değerini sade Türkçe ile nasıl yorumlarsınız?

<details>
    <summary>Cevap</summary>

> "Diğer tüm şeyler eşitken (yaş, bilet sınıfı vb...),
kadın olmak hayatta kalma log-oranlarınızı 2.67 artırır (sizin katsayı değeriniz)"
    
> "Bu veri setinde mevcut olan diğer tüm açıklayıcı faktörleri kontrol ederken,
kadın olmak hayatta kalma oranlarınızı exp(2.67) = 14 kat artırır"

</details>

In [8]:
coef_female = coefs_unreg["sex_female"]
print(f"sex_female katsayısı: {coef_female:.2f}")
print(f"Odds çarpanı: exp({coef_female:.2f}) = {np.exp(coef_female):.1f} kat")

sex_female katsayısı: 2.67
Odds çarpanı: exp(2.67) = 14.5 kat


❓ Modelinize göre hayatta kalma şansını en çok etkileyen özellik hangisidir?  
Aşağıdaki `top_1_feature` listesini bu özelliğin adıyla doldurun

In [9]:
top_1_feature = [coefs_unreg.abs().idxmax()]
top_1_feature

['embark_town_Queenstown']

In [10]:
from nbresult import ChallengeResult
result = ChallengeResult('unregularized', top_1_feature=top_1_feature)
result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-9.1.1, pluggy-1.5.0 -- /home/gokberk/.pyenv/versions/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/gokberk/code/gokberkozan/S16D5-S-regularization/tests
plugins: anyio-4.8.0, typeguard-4.4.2
collecting ... collected 1 item

test_unregularized.py::TestUnregularized::test_top_1 PASSED              [100%]

============================== 1 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/unregularized.pickle

git commit -m 'Completed unregularized step'

git push origin master



## 3. L2 cezalı Lojistik Regresyon

Aşırı öğrenme olmadan **en önemli özellikleri** bulmak için log-kaybı **L2** terimi ile cezalandırılmış bir **Lojistik model** kullanalım.  
Bu, "Ridge" regresörünün "sınıflandırma" karşılığıdır

❓ **Güçlü düzenlileştirilmiş** bir `LogisticRegression` oluşturun ve özelliklerini önem sırasına göre sıralayın (katsayılara bakın)
- "Güçlü düzenlileştirilmiş" ile "Sklearn'in varsayılan düzenlileştirme faktöründen daha fazla" demek istiyoruz. 
- Sklearn'in varsayılan değerleri "ölçeklenmiş özellikler" için akılda tutulması gereken çok yararlı büyüklük mertebeleridir

In [11]:
log_l2 = LogisticRegression(penalty="l2", C=0.1, max_iter=10000)
log_l2.fit(X_scaled, y)

coefs_l2 = pd.Series(log_l2.coef_[0], index=X.columns)
coefs_l2.reindex(coefs_l2.abs().sort_values(ascending=False).index)

sex_female                 1.808312
class_Third               -0.637598
who_child                  0.603138
pclass                    -0.539197
age                       -0.478735
class_First                0.441382
sibsp                     -0.341013
embark_town_Cherbourg      0.252479
embark_town_Southampton   -0.154604
fare                       0.137767
embark_town_Queenstown    -0.131398
parch                     -0.053818
dtype: float64

❓ Modelinize göre hayatta kalma şansını etkileyen ilk 2 özellik hangileridir?  
Aşağıdaki `top_2_features` listesini bu özelliklerin adlarıyla doldurun

In [12]:
top_2_features = coefs_l2.abs().sort_values(ascending=False).index[:2].tolist()
top_2_features

['sex_female', 'class_Third']

#### 🧪 Kodunuzu aşağıda test edin

In [13]:
from nbresult import ChallengeResult
result = ChallengeResult('ridge', top_2=top_2_features)
result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-9.1.1, pluggy-1.5.0 -- /home/gokberk/.pyenv/versions/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/gokberk/code/gokberkozan/S16D5-S-regularization/tests
plugins: anyio-4.8.0, typeguard-4.4.2
collecting ... collected 1 item

test_ridge.py::TestRidge::test_top2 PASSED                               [100%]

============================== 1 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/ridge.pickle

git commit -m 'Completed ridge step'

git push origin master



## 4. L1 cezalı Lojistik Regresyon

Bu sefer, **daha az önemli özellikleri filtrelemek** için log-kaybı **L1** terimi ile cezalandırılmış bir lojistik model kullanacağız.  
Bu, **Lasso** regresörünün "sınıflandırma" karşılığıdır

❓ **Güçlü düzenlileştirilmiş** bir `LogisticRegression` oluşturun ve özelliklerini önem sırasına göre sıralayın

In [14]:
log_l1 = LogisticRegression(penalty="l1", solver="liblinear", C=0.1, max_iter=10000)
log_l1.fit(X_scaled, y)

coefs_l1 = pd.Series(log_l1.coef_[0], index=X.columns)
coefs_l1.reindex(coefs_l1.abs().sort_values(ascending=False).index)

sex_female                 2.000616
pclass                    -1.443907
who_child                  0.255963
embark_town_Southampton   -0.239097
class_Third               -0.143617
age                       -0.062863
sibsp                      0.000000
parch                      0.000000
fare                       0.000000
class_First                0.000000
embark_town_Cherbourg      0.000000
embark_town_Queenstown     0.000000
dtype: float64

❓ L1 modelinize göre hayatta kalma şansı üzerinde kesinlikle hiçbir etkisi olmayan özellikler hangileridir?  
Aşağıdaki `zero_impact_features` listesini bu özelliklerin adlarıyla doldurun; listeye eleman eklemeniz gerekebilir.

- Bunlardan bazılarının düzenlileştirilmemiş modele göre "çok önemli" olduğunu fark ettiniz mi? 
- Bundan sonra doğrusal modellerimizi her zaman düzenlileştireceğiz!

In [15]:
zero_impact_features = coefs_l1[coefs_l1 == 0].index.tolist()
zero_impact_features

['sibsp',
 'parch',
 'fare',
 'class_First',
 'embark_town_Cherbourg',
 'embark_town_Queenstown']

#### 🧪 Kodunuzu aşağıda test edin

In [16]:
from nbresult import ChallengeResult
result = ChallengeResult('lasso', zero_impact_features = zero_impact_features)
result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-9.1.1, pluggy-1.5.0 -- /home/gokberk/.pyenv/versions/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/gokberk/code/gokberkozan/S16D5-S-regularization/tests
plugins: anyio-4.8.0, typeguard-4.4.2
collecting ... collected 1 item

test_lasso.py::TestLasso::test_zero_impact PASSED                        [100%]

============================== 1 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/lasso.pickle

git commit -m 'Completed lasso step'

git push origin master



Farklı C değerlerini dene

In [17]:
from sklearn.model_selection import cross_val_score

rows = []
for C in [0.01, 0.05, 0.1, 0.2, 0.5, 1, 5, 10]:
    model = LogisticRegression(penalty="l1", solver="liblinear", C=C, max_iter=10000)
    acc = cross_val_score(model, X_scaled, y, cv=5, scoring="accuracy").mean()
    n_zero = (model.fit(X_scaled, y).coef_[0] == 0).sum()
    rows.append({"C": C, "alpha (1/C)": 1 / C, "CV accuracy": round(acc, 4), "sıfır katsayı": int(n_zero)})

c_search = pd.DataFrame(rows)
best_C = c_search.loc[c_search["CV accuracy"].idxmax(), "C"]
print("En iyi C:", best_C)
c_search

En iyi C: 5.0


,C,alpha (1/C),CV accuracy,sıfır katsayı
0,0.01,100.0,0.5938,12
1,0.05,20.0,0.7801,8
2,0.10,10.0,0.7801,6
3,0.20,5.0,0.7731,4
4,0.50,2.0,0.7844,4
5,1.00,1.0,0.7928,2
6,5.00,0.2,0.7984,1
7,10.00,0.1,0.7984,1


Optimal modelle feature seçimi

In [18]:
best_l1 = LogisticRegression(penalty="l1", solver="liblinear", C=best_C, max_iter=10000).fit(X_scaled, y)
coefs_best = pd.Series(best_l1.coef_[0], index=X.columns)

selected = coefs_best[coefs_best != 0].index.tolist()
eliminated = coefs_best[coefs_best == 0].index.tolist()
print("Seçilen feature'lar :", selected)
print("Elenen feature'lar  :", eliminated)

acc_all = cross_val_score(LogisticRegression(max_iter=10000), X_scaled, y, cv=5).mean()
acc_sel = cross_val_score(LogisticRegression(max_iter=10000), X_scaled[selected], y, cv=5).mean()
print(f"\nTüm feature'lar ({X.shape[1]})    -> CV accuracy: {acc_all:.4f}")
print(f"Seçilen feature'lar ({len(selected)}) -> CV accuracy: {acc_sel:.4f}")

Seçilen feature'lar : ['pclass', 'age', 'sibsp', 'parch', 'fare', 'sex_female', 'class_First', 'class_Third', 'who_child', 'embark_town_Cherbourg', 'embark_town_Queenstown']
Elenen feature'lar  : ['embark_town_Southampton']

Tüm feature'lar (12)    -> CV accuracy: 0.7942
Seçilen feature'lar (11) -> CV accuracy: 0.7942


# 5. Bir adım geri çekilmek

🤯 **Bu katsayılardan bazıları neden başlangıçta bu kadar yüksekti?**

Düzenlileştirme ile kaldırılan üç özelliği düşünelim:
- `embark_town_Cherbourg`
- `embark_town_Southampton`
- `embark_town_Queenstown`

Üç biniş şehri tabii ki ilişkilidir: ikisinden binmediyseniz, üçüncüsünden binmiş olmalısınız. Yani biliyoruz ki: 

$$embark\_town\_Cherbourg + embark\_town\_Southampton + embark\_town\_Queenstown = 1$$

Bu üç özellik **mükemmel çoklu doğrusal bağıntılıdır**!

**Düzenlileştirilmemiş modeller kullanılırken, bu genellikle sayısal kararsızlığa yol açar**, ki burada gördüğümüz tam olarak buydu. Ayrıca böyle bir durumda elde ettiğimiz **katsayılara gerçekten güvenemeyeceğimiz** anlamına gelir.

❗️ Bu üç çoklu doğrusal bağıntılı özellik, `embark_town` kategorik özelliğinin one hot encoding'inden gelir.

Düzenlileştirme sayesinde bu sorunu aştık: üç şehir için katsayıların çok büyük olmasını engelledi. **İşte bu yüzden neredeyse her zaman düzenlileştirme kullanacağız.**

🔍 **Başlangıçta ayarladığımız `tol` parametresini hatırlıyor musunuz?**

Düzenlileştirmenin ekstra bir bonusu da `tol` ayarlamanın daha az önemli hale gelmesi: `1e-2` ve `1e-9` arasında herhangi bir değere ayarlayabilirsiniz ve katsayılar neredeyse hiç değişmez! 💪

## Yorum

- **Cezasız model:** En büyük katsayılar `embark_town_*` sütunlarında çıktı. Bu üç sütun one-hot encoding'den geldiği için toplamları her zaman 1 (mükemmel çoklu doğrusallık) → katsayılar şişip kararsızlaşıyor, güvenilir değil.
- **L2 (Ridge, C=0.1):** Katsayıları küçülterek bu şişmeyi bastırdı; en önemli feature'lar mantıklı olanlar oldu (cinsiyet, yolcu sınıfı, çocuk olmak).
- **L1 (Lasso):** Önemsiz feature'ların katsayısını tam 0 yaptı; cezasız modelde "en önemli" görünen `embark_town_*` sütunları burada elendi.
- **Parametre araması:** L1 için farklı C (= 1/alpha) değerleri 5-fold CV ile denendi; en yüksek CV accuracy veren C seçildi. C küçüldükçe daha fazla feature eleniyor, accuracy ise belli bir noktaya kadar neredeyse aynı kalıyor.
- **Feature selection:** Optimal Lasso modelinin sıfır olmayan katsayıları seçilen feature setini oluşturdu. Bu daha küçük setle elde edilen CV accuracy, tüm feature'larla elde edilene çok yakın → elenen feature'lar modele anlamlı bilgi katmıyordu.

**🏁 Tebrikler! Not defterinizi commit etmeyi ve push etmeyi unutmayın**